In [ ]:
# =====================================================================
# 🏦 PROJECT: BANK CUSTOMER CHURN PREDICTOR (FINAL REVISED CODE)
# =====================================================================

import numpy as np       
import pandas as pd      
import torch            
import torch.nn as nn   
import torch.optim as optim  
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler      

# Seed set करना ताकि हर बार रिजल्ट सेम आए
np.random.seed(42) 
n_samples = 5000

# डेटा डिक्शनरी (ब्रैकेट्स [0, 1] को यहाँ ठीक कर दिया गया है)
data = {
    'Credit_Score': np.random.randint(300, 850, size=n_samples), 
    'Age': np.random.randint(18, 70, size=n_samples),           
    'Balance': np.random.uniform(1000, 100000, size=n_samples), 
    'Is_Active': np.random.choice([0, 1], size=n_samples),       
    'Churn': np.random.choice([0, 1], size=n_samples, p=[0.8, 0.2]) 
}

df = pd.DataFrame(data) 

# ---- डेटा प्रीप्रोसेसिंग ----
X = df.drop('Churn', axis=1).values 
y = df['Churn'].values             

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train) 
X_test = scaler.transform(X_test)       

# NumPy से PyTorch Tensors में बदलना
X_train_tensor = torch.FloatTensor(X_train)
y_train_tensor = torch.FloatTensor(y_train).unsqueeze(1) 
X_test_tensor = torch.FloatTensor(X_test)
y_test_tensor = torch.FloatTensor(y_test).unsqueeze(1)

# ---- मॉडल आर्किटेक्चर ----
class BankChurnModel(nn.Module):
    def __init__(self, input_dim):
        super(BankChurnModel, self).__init__()
        self.layer1 = nn.Linear(input_dim, 16)
        self.relu1 = nn.ReLU() 
        self.dropout = nn.Dropout(p=0.2) 
        self.layer2 = nn.Linear(16, 8)
        self.relu2 = nn.ReLU()
        self.output_layer = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu1(x)
        x = self.dropout(x) 
        x = self.layer2(x)
        x = self.relu2(x)
        x = self.output_layer(x)
        x = self.sigmoid(x) 
        return x

model = BankChurnModel(input_dim=4)

# ---- लॉस और ऑप्टिमाइज़र ----
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.005)

# ---- ट्रेनिंग लूप ----
epochs = 100 
for epoch in range(epochs):
    model.train() 
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor) 
    
    optimizer.zero_grad() 
    loss.backward()       
    optimizer.step()      
    
    if (epoch+1) % 20 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# ---- इवैल्यूएशन (टेस्ट) ----
model.eval() 
with torch.no_grad(): 
    test_outputs = model(X_test_tensor)
    predictions = (test_outputs >= 0.5).float()
    correct = (predictions == y_test_tensor).float().sum()
    accuracy = (correct / y_test_tensor.shape[0]) * 100
    print(f"\n🎯 FINAL TEST ACCURACY: {accuracy.item():.2f}%")


Epoch [20/100], Loss: 0.4994
Epoch [40/100], Loss: 0.4943
Epoch [60/100], Loss: 0.4935
Epoch [80/100], Loss: 0.4917
Epoch [100/100], Loss: 0.4909

🎯 FINAL TEST ACCURACY: 79.90%


In [ ]:
import numpy as np 
import pandas as pd 
import tourch
import tourch.nn as nn 
